# Module 4 — Mathematics for Scientific Computing & Machine Learning
## Hands-On Python Tutorial for Applied / Computational Materials

**Level:** IIT M.Tech / PhD  
**Format:** 3-hour tutorial  
**Prerequisites:** Modules 1–3

### Learning objectives

By the end of this tutorial, you should be able to:

- Analyze scientific data using descriptive statistics and probability.
- Calculate covariance and correlation and interpret them critically.
- Implement numerical differentiation and integration.
- Solve nonlinear equations using root-finding methods.
- Interpolate experimental materials data.
- Perform least-squares parameter estimation.
- Fit physically motivated models such as the Arrhenius equation.
- Formulate and solve optimization problems using SciPy.
- Understand why least squares and optimization are foundations of machine learning.

> **Core philosophy:** equation → numerical method → Python implementation → verification → physical interpretation.


## Tutorial roadmap

| Section | Topic | Materials connection |
|---|---|---|
| 1 | Scientific Python setup | Reproducible computation |
| 2 | Descriptive statistics | Experimental materials data |
| 3 | Probability | Measurement uncertainty |
| 4 | Covariance & correlation | Materials descriptors |
| 5 | Numerical differentiation | Stress–strain / property curves |
| 6 | Numerical integration | Work, energy, heat |
| 7 | Root finding | Equilibrium conditions |
| 8 | Interpolation | Property lookup |
| 9 | Least squares | Parameter estimation |
| 10 | Curve fitting | Arrhenius diffusion |
| 11 | Optimization | Parameter calibration |
| 12 | Integrated problem | Diffusion activation energy |
| 13 | Exercises and assignment | Independent practice |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import integrate, optimize, interpolate, stats

np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(42)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


# 1. Descriptive Statistics

Suppose repeated hardness measurements are obtained:

\[
H=[201,205,198,210,203,207,202,199,211,205]\;HV.
\]

The mean is

\[
\bar{x}=\frac{1}{N}\sum_i x_i
\]

and sample variance is

\[
s^2=\frac{1}{N-1}\sum_i(x_i-\bar{x})^2.
\]

The sample standard deviation is \(s=\sqrt{s^2}\).


In [ ]:
hardness = np.array([201, 205, 198, 210, 203, 207, 202, 199, 211, 205])

print("Mean =", np.mean(hardness))
print("Median =", np.median(hardness))
print("Sample standard deviation =", np.std(hardness, ddof=1))
print("Sample variance =", np.var(hardness, ddof=1))
print("Minimum =", np.min(hardness))
print("Maximum =", np.max(hardness))


## Exercise 1 — Grain-size statistics

Create 15 grain-size measurements in μm and calculate:

1. Mean
2. Median
3. Standard deviation
4. Minimum and maximum
5. Range
6. 25th and 75th percentiles

Then plot a histogram and comment on the distribution.


In [ ]:
grain_size = np.array([
    8.2, 9.1, 8.7, 10.2, 9.8,
    11.0, 8.9, 9.4, 10.1, 9.6,
    8.5, 12.0, 10.4, 9.2, 8.8
])

print("Mean:", np.mean(grain_size))
print("Median:", np.median(grain_size))
print("Std:", np.std(grain_size, ddof=1))
print("Min:", np.min(grain_size))
print("Max:", np.max(grain_size))
print("Range:", np.ptp(grain_size))
print("25th percentile:", np.percentile(grain_size, 25))
print("75th percentile:", np.percentile(grain_size, 75))

plt.figure(figsize=(7,4))
plt.hist(grain_size, bins=6, edgecolor="black")
plt.xlabel("Grain size (µm)")
plt.ylabel("Frequency")
plt.title("Grain-size distribution")
plt.show()


# 2. Probability and Measurement Uncertainty

A useful measurement model is

\[
x_{\mathrm{measured}}=x_{\mathrm{true}}+\epsilon
\]

with

\[
\epsilon\sim N(0,\sigma^2).
\]

We can simulate repeated experimental measurements and study their distribution.


In [ ]:
true_value = 200.0
sigma = 5.0
N = 1000

measurements = true_value + rng.normal(0, sigma, N)

print("Measured mean:", np.mean(measurements))
print("Measured std:", np.std(measurements, ddof=1))

plt.figure(figsize=(7,4))
plt.hist(measurements, bins=30, edgecolor="black")
plt.axvline(true_value, linestyle="--", label="True value")
plt.xlabel("Measured property")
plt.ylabel("Frequency")
plt.title("Simulated measurement uncertainty")
plt.legend()
plt.show()


## Normal distribution with SciPy

The normal probability density is

\[
f(x)=\frac{1}{\sigma\sqrt{2\pi}}
\exp\left[-\frac{(x-\mu)^2}{2\sigma^2}\right].
\]

Use `scipy.stats.norm` to calculate the PDF, CDF, and probabilities.


In [ ]:
mu, sigma = 200, 5
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 400)

pdf = stats.norm.pdf(x, loc=mu, scale=sigma)

plt.figure(figsize=(7,4))
plt.plot(x, pdf)
plt.xlabel("Property value")
plt.ylabel("Probability density")
plt.title("Normal distribution")
plt.grid()
plt.show()

p = stats.norm.cdf(205, mu, sigma) - stats.norm.cdf(195, mu, sigma)
print(f"P(195 < X < 205) = {p:.4f}")


# 3. Covariance and Correlation

Covariance is

\[
\operatorname{Cov}(X,Y)
=
\frac{1}{N-1}\sum_i(x_i-\bar{x})(y_i-\bar{y}).
\]

Pearson correlation is

\[
r=\frac{\operatorname{Cov}(X,Y)}{s_Xs_Y}.
\]

Interpretation:

- \(r\approx+1\): strong positive linear relationship
- \(r\approx-1\): strong negative linear relationship
- \(r\approx0\): weak linear relationship

**Important:** correlation does not imply causation.


In [ ]:
grain = np.array([5, 7, 9, 11, 13, 15, 17, 19], dtype=float)
hardness2 = np.array([240, 226, 215, 205, 196, 188, 181, 175], dtype=float)

print("Covariance matrix:")
print(np.cov(grain, hardness2))

print("\nCorrelation matrix:")
print(np.corrcoef(grain, hardness2))

plt.figure(figsize=(7,4))
plt.scatter(grain, hardness2)
plt.xlabel("Grain size (µm)")
plt.ylabel("Hardness (HV)")
plt.title("Grain size versus hardness")
plt.grid()
plt.show()


## Exercise 3 — Hidden-variable correlation

Generate

\[
x=2z+\epsilon_x,\qquad
y=-3z+\epsilon_y
\]

where \(z\) is a common hidden variable. Calculate the correlation and explain why a strong correlation alone does not prove a direct physical causal relationship.


In [ ]:
z = rng.normal(size=100)
x = 2*z + rng.normal(0, 0.4, 100)
y = -3*z + rng.normal(0, 0.6, 100)

print("Correlation =", np.corrcoef(x, y)[0,1])


# 4. Numerical Differentiation

For

\[
f'(x)=\lim_{h\rightarrow0}\frac{f(x+h)-f(x)}{h},
\]

a forward difference is

\[
f'(x)\approx\frac{f(x+h)-f(x)}{h},
\]

while a central difference is

\[
f'(x)\approx\frac{f(x+h)-f(x-h)}{2h}.
\]

We compare numerical differentiation with the exact derivative of

\[
f(x)=\sin x,\qquad f'(x)=\cos x.
\]


In [ ]:
x = np.linspace(0, 2*np.pi, 200)
f = np.sin(x)
h = x[1] - x[0]

numerical = np.gradient(f, h, edge_order=2)
exact = np.cos(x)

print("Maximum absolute error =", np.max(np.abs(numerical-exact)))

plt.figure(figsize=(8,4))
plt.plot(x, exact, label="Exact")
plt.plot(x, numerical, "--", label="Numerical")
plt.xlabel("x")
plt.ylabel("df/dx")
plt.title("Numerical differentiation")
plt.legend()
plt.grid()
plt.show()


## Materials application — tangent modulus

The tangent modulus is

\[
E_t=\frac{d\sigma}{d\epsilon}.
\]

Given discrete stress–strain data, estimate \(E_t\) numerically.


In [ ]:
strain = np.linspace(0, 0.02, 100)
stress = 70000*strain - 8e5*strain**2

tangent_modulus = np.gradient(stress, strain)

plt.figure(figsize=(7,4))
plt.plot(strain, stress)
plt.xlabel("Strain")
plt.ylabel("Stress (MPa)")
plt.title("Stress-strain curve")
plt.grid()
plt.show()

plt.figure(figsize=(7,4))
plt.plot(strain, tangent_modulus)
plt.xlabel("Strain")
plt.ylabel("Tangent modulus (MPa)")
plt.title("Numerical tangent modulus")
plt.grid()
plt.show()


# 5. Numerical Integration

The definite integral

\[
I=\int_a^b f(x)\,dx
\]

represents accumulated quantity or area.

Materials examples include:

- work from stress–strain data
- heat absorbed
- enthalpy changes
- integrated concentration
- total reaction amount

For

\[
\int_0^1x^2dx=\frac13
\]

we can compare numerical and analytical results.


In [ ]:
x = np.linspace(0, 1, 101)
y = x**2

I = np.trapezoid(y, x)

print("Numerical integral =", I)
print("Exact integral =", 1/3)
print("Absolute error =", abs(I-1/3))


In [ ]:
def f(x):
    return np.exp(-x**2)

result, error = integrate.quad(f, 0, 1)

print("SciPy integral =", result)
print("Estimated numerical error =", error)


## Materials application — mechanical work

For uniaxial loading,

\[
W=\int\sigma\,d\epsilon.
\]

Calculate the work per unit volume from the stress–strain curve above.


In [ ]:
work_density = np.trapezoid(stress, strain)
print(f"Work density = {work_density:.4f} MPa")
print("Numerically, MPa is equivalent to MJ/m³.")


# 6. Root Finding

Many scientific models require solving

\[
f(x)=0.
\]

Examples include equilibrium, critical conditions, intersections, and nonlinear constitutive equations.

Consider

\[
f(x)=x^3-2x-5.
\]


In [ ]:
def f_root(x):
    return x**3 - 2*x - 5

x_plot = np.linspace(-3, 3, 400)

plt.figure(figsize=(7,4))
plt.plot(x_plot, f_root(x_plot))
plt.axhline(0, linestyle="--")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Root-finding problem")
plt.grid()
plt.show()


## Implement bisection

If \(f(a)f(b)<0\), a continuous function has at least one root in \([a,b]\).

Implement the bisection algorithm and stop when either the residual or interval width is below the tolerance.


In [ ]:
def bisection(f, a, b, tol=1e-10, max_iter=100):
    fa, fb = f(a), f(b)
    if fa * fb > 0:
        raise ValueError("Interval does not bracket a root.")

    for _ in range(max_iter):
        c = 0.5*(a+b)
        fc = f(c)

        if abs(fc) < tol or abs(b-a) < tol:
            return c

        if fa*fc < 0:
            b, fb = c, fc
        else:
            a, fa = c, fc

    return 0.5*(a+b)

root = bisection(f_root, 2, 3)
print("Bisection root =", root)
print("Residual =", f_root(root))


In [ ]:
root_scipy = optimize.brentq(f_root, 2, 3)
print("SciPy root =", root_scipy)
print("Residual =", f_root(root_scipy))


# 7. Interpolation

Experimental data are discrete. Interpolation estimates values between measured points.

Suppose thermal conductivity is measured at selected temperatures.


In [ ]:
T_data = np.array([300, 400, 500, 600, 700])
k_data = np.array([220, 185, 155, 132, 115])

T_query = np.linspace(300, 700, 200)

linear = interpolate.interp1d(T_data, k_data, kind="linear")
cubic = interpolate.interp1d(T_data, k_data, kind="cubic")

plt.figure(figsize=(8,4))
plt.scatter(T_data, k_data, label="Measurements")
plt.plot(T_query, linear(T_query), label="Linear")
plt.plot(T_query, cubic(T_query), label="Cubic")
plt.xlabel("Temperature (K)")
plt.ylabel("Thermal conductivity (W/m/K)")
plt.title("Interpolation")
plt.legend()
plt.grid()
plt.show()


### Scientific caution

**Interpolation** estimates within the measured range.

**Extrapolation** estimates outside it and can be unreliable, especially for nonlinear materials behavior.

Always identify the data range before using an interpolant.


# 8. Least Squares

Suppose

\[
y=ax+b.
\]

We choose \(a,b\) to minimize

\[
J(a,b)=\sum_i[y_i-(ax_i+b)]^2.
\]

In matrix form,

\[
y=X\beta.
\]

The normal-equation form is

\[
X^TX\hat{\beta}=X^Ty.
\]

Although

\[
\hat{\beta}=(X^TX)^{-1}X^Ty
\]

is useful mathematically, stable numerical implementations generally use QR/SVD-based methods rather than explicitly forming the inverse.


In [ ]:
x = np.linspace(0, 10, 30)
y_true = 3.2*x + 2
y = y_true + rng.normal(0, 1.5, x.size)

X = np.column_stack([x, np.ones_like(x)])
beta, residuals, rank, singular_values = np.linalg.lstsq(X, y, rcond=None)

a, b = beta
y_fit = a*x+b

print("Slope =", a)
print("Intercept =", b)
print("Rank =", rank)
print("SSE =", residuals[0])

plt.figure(figsize=(7,4))
plt.scatter(x, y, label="Data")
plt.plot(x, y_fit, label="Least-squares fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Least-squares regression")
plt.legend()
plt.grid()
plt.show()


# 9. Curve Fitting — Arrhenius Diffusion

Diffusion commonly follows

\[
D=D_0\exp\left(-\frac{Q}{RT}\right).
\]

Taking logarithms:

\[
\ln D=\ln D_0-\frac{Q}{R}\frac{1}{T}.
\]

Therefore, a plot of \(\ln D\) against \(1/T\) is approximately linear.

This connects:

\[
\boxed{\text{materials physics}\rightarrow\text{transformation}\rightarrow\text{least squares}\rightarrow\text{parameter extraction}}
\]


In [ ]:
R = 8.314
T = np.array([500,550,600,650,700,750,800,850], dtype=float)

D0_true = 2e-6
Q_true = 85000

D_true = D0_true*np.exp(-Q_true/(R*T))
D_measured = D_true*np.exp(rng.normal(0, 0.08, T.size))

slope, intercept = np.polyfit(1/T, np.log(D_measured), 1)

Q_fit = -slope*R
D0_fit = np.exp(intercept)

print(f"True Q = {Q_true/1000:.2f} kJ/mol")
print(f"Fit Q = {Q_fit/1000:.2f} kJ/mol")
print(f"True D0 = {D0_true:.3e} m²/s")
print(f"Fit D0 = {D0_fit:.3e} m²/s")


In [ ]:
x_fit = np.linspace((1/T).min(), (1/T).max(), 200)
y_fit = slope*x_fit+intercept

plt.figure(figsize=(7,4))
plt.scatter(1/T, np.log(D_measured), label="Data")
plt.plot(x_fit, y_fit, label="Linear fit")
plt.xlabel("1/T (1/K)")
plt.ylabel("ln(D)")
plt.title("Arrhenius analysis")
plt.legend()
plt.grid()
plt.show()


## Exercise — Interpret the Arrhenius fit

Answer:

1. What is the physical meaning of \(Q\)?
2. What is the physical meaning of \(D_0\)?
3. Why does the logarithmic transformation produce a straight line?
4. What assumptions are involved in fitting a single activation energy?
5. Why might real data deviate from a straight line?


# 10. Optimization

Optimization solves

\[
\theta^*=\arg\min_\theta J(\theta).
\]

Applications include:

- parameter calibration
- energy minimization
- process optimization
- inverse modelling
- fitting constitutive models

Consider

\[
J(x)=(x-3)^2+2.
\]


In [ ]:
def objective(x):
    return (x-3)**2 + 2

result = optimize.minimize_scalar(objective)

print("Optimal x =", result.x)
print("Minimum J =", result.fun)
print("Success =", result.success)


In [ ]:
x = np.linspace(-2, 8, 300)

plt.figure(figsize=(7,4))
plt.plot(x, objective(x))
plt.axvline(result.x, linestyle="--", label="Optimum")
plt.xlabel("x")
plt.ylabel("J(x)")
plt.title("Optimization")
plt.legend()
plt.grid()
plt.show()


# 11. Nonlinear parameter calibration

Consider a simplified stress model

\[
\sigma(\epsilon)=E\epsilon+A\epsilon^2.
\]

We can estimate \(E,A\) by minimizing

\[
J(E,A)=\sum_i[\sigma_i-\sigma_{\mathrm{model},i}]^2.
\]

This is the same basic optimization idea used in many inverse materials problems.


In [ ]:
strain = np.linspace(0, 0.02, 50)

E_true = 70000
A_true = -900000

stress_true = E_true*strain + A_true*strain**2
stress_exp = stress_true + rng.normal(0, 8, strain.size)

def stress_model(params, strain):
    E, A = params
    return E*strain + A*strain**2

def objective_params(params):
    residual = stress_exp - stress_model(params, strain)
    return np.sum(residual**2)

result = optimize.minimize(
    objective_params,
    x0=np.array([50000.0, 0.0])
)

E_fit, A_fit = result.x

print("True E =", E_true)
print("Fitted E =", E_fit)
print("True A =", A_true)
print("Fitted A =", A_fit)
print("Objective =", result.fun)


In [ ]:
stress_fit = stress_model(result.x, strain)

plt.figure(figsize=(7,4))
plt.scatter(strain, stress_exp, label="Synthetic experiment")
plt.plot(strain, stress_fit, label="Optimized model")
plt.xlabel("Strain")
plt.ylabel("Stress (MPa)")
plt.title("Parameter calibration")
plt.legend()
plt.grid()
plt.show()


# 12. Connection to Machine Learning

For linear regression,

\[
\hat{y}=X\beta
\]

and the mean squared error is

\[
J(\beta)=\frac{1}{N}\sum_i(y_i-\hat{y}_i)^2.
\]

Training is therefore an optimization problem:

\[
\beta^*=\arg\min_\beta J(\beta).
\]

This gives the important progression:

\[
\boxed{
\text{statistics}
\rightarrow
\text{least squares}
\rightarrow
\text{objective functions}
\rightarrow
\text{optimization}
\rightarrow
\text{machine learning}
}
\]

Understanding this chain is more important than memorizing `model.fit()`.


# 13. Integrated Materials Problem — Diffusion Activation Energy

Use the Arrhenius model

\[
D=D_0e^{-Q/(RT)}
\]

to estimate \(D_0\) and \(Q\).

You will:

1. Inspect the dataset.
2. Plot \(D(T)\).
3. Transform to \(\ln D\) versus \(1/T\).
4. Fit the linearized model.
5. Extract \(Q,D_0\).
6. Fit the model directly using optimization.
7. Calculate residuals.
8. Compare the approaches.
9. Interpret the physics.


In [ ]:
R = 8.314

T_exp = np.array([450,500,550,600,650,700,750,800,850], dtype=float)
D0_true = 5e-7
Q_true = 92000

D_exp = D0_true*np.exp(-Q_true/(R*T_exp))
D_exp *= np.exp(rng.normal(0, 0.06, T_exp.size))

data = pd.DataFrame({
    "Temperature_K": T_exp,
    "Diffusivity_m2_s": D_exp
})

data


In [ ]:
plt.figure(figsize=(7,4))
plt.semilogy(T_exp, D_exp, "o-")
plt.xlabel("Temperature (K)")
plt.ylabel("Diffusivity (m²/s)")
plt.title("Diffusivity versus temperature")
plt.grid()
plt.show()


In [ ]:
slope, intercept = np.polyfit(1/T_exp, np.log(D_exp), 1)

Q_linear = -slope*R
D0_linear = np.exp(intercept)

print(f"Linearized fit: Q = {Q_linear/1000:.2f} kJ/mol")
print(f"Linearized fit: D0 = {D0_linear:.3e} m²/s")


## Direct optimization in transformed parameter space

To improve numerical scaling, optimize \(\ln D_0\) rather than \(D_0\) itself:

\[
\ln D=\ln D_0-\frac{Q}{RT}.
\]

Minimize the squared residual in log-space.


In [ ]:
def arrhenius_log_model(params, T):
    log_D0, Q = params
    return log_D0 - Q/(R*T)

def arrhenius_objective(params):
    residual = np.log(D_exp) - arrhenius_log_model(params, T_exp)
    return np.sum(residual**2)

initial = np.array([np.log(1e-6), 80000.0])

opt_result = optimize.minimize(arrhenius_objective, initial)

log_D0_opt, Q_opt = opt_result.x
D0_opt = np.exp(log_D0_opt)

print(f"Optimized Q = {Q_opt/1000:.2f} kJ/mol")
print(f"Optimized D0 = {D0_opt:.3e} m²/s")
print("Optimization successful:", opt_result.success)


In [ ]:
D_fit = D0_opt*np.exp(-Q_opt/(R*T_exp))
residuals = D_exp-D_fit

rmse = np.sqrt(np.mean(residuals**2))

print("Residuals:")
print(residuals)
print("\nRMSE =", rmse)


In [ ]:
T_smooth = np.linspace(T_exp.min(), T_exp.max(), 300)
D_smooth = D0_opt*np.exp(-Q_opt/(R*T_smooth))

plt.figure(figsize=(8,4))
plt.semilogy(T_exp, D_exp, "o", label="Data")
plt.semilogy(T_smooth, D_smooth, label="Fitted Arrhenius model")
plt.xlabel("Temperature (K)")
plt.ylabel("Diffusivity (m²/s)")
plt.title("Diffusion activation-energy fit")
plt.legend()
plt.grid()
plt.show()


# 14. Numerical accuracy — finite-difference step size

Numerical differentiation contains approximation error.

For central differences, decreasing \(h\) initially improves accuracy, but extremely small \(h\) can suffer from floating-point round-off.

Study

\[
f(x)=\sin x
\]

at \(x=1\).


In [ ]:
x0 = 1.0
exact = np.cos(x0)

hs = np.logspace(-1, -12, 12)
errors = []

for h in hs:
    numerical = (np.sin(x0+h)-np.sin(x0-h))/(2*h)
    errors.append(abs(numerical-exact))

plt.figure(figsize=(7,4))
plt.loglog(hs, errors, marker="o")
plt.xlabel("Step size h")
plt.ylabel("Absolute error")
plt.title("Finite-difference error")
plt.grid()
plt.show()

i = np.argmin(errors)
print("Best h tested =", hs[i])
print("Minimum error =", errors[i])


# 15. Hands-on exercises

### Exercise A — Thermal expansion

Given

\[
L(T)=L_0[1+\alpha(T-T_0)],
\]

calculate \(dL/dT\) numerically and compare with the analytical result.

### Exercise B — Heat capacity

Given experimental \(C_p(T)\), calculate

\[
Q=\int_{T_1}^{T_2}C_p(T)dT.
\]

Compare trapezoidal integration with SciPy.

### Exercise C — Equilibrium

Define

\[
G(x)=x^4-2x^2+x.
\]

Find stationary points by solving

\[
dG/dx=0.
\]

Classify each stationary point.

### Exercise D — Polynomial fitting

Generate noisy data from

\[
y=2+3x-0.5x^2.
\]

Fit polynomials of degree 1, 2, 5, and 10. Compare fit quality and discuss overfitting.

### Exercise E — Materials-property correlations

Create a DataFrame containing at least five synthetic descriptors and a target property. Calculate covariance, correlation, and identify the strongest relationships.


# 16. Challenge — Optimize a processing condition

Consider

\[
P(T,t)=P_0+
A\exp\left(-\frac{Q}{RT}\right)
(1-e^{-kt}).
\]

Choose \(T,t\) to maximize \(P\), subject to

\[
500\le T\le1000\;K,\qquad
1\le t\le100\;s.
\]

### Tasks

1. Define the model.
2. Define the objective.
3. Use `scipy.optimize.minimize`.
4. Apply bounds.
5. Report the optimum.
6. Plot \(P(T,t)\).
7. Discuss why a mathematical optimum is not automatically a physically optimal manufacturing condition.


# 17. Mini-project — Scientific parameter estimation

Choose one:

### A. Diffusion
Fit an Arrhenius model and estimate activation energy.

### B. Grain growth
Fit

\[
d^n-d_0^n=Kt.
\]

### C. Thermal conductivity
Fit a temperature-dependent model.

### D. Stress–strain
Estimate elastic and nonlinear parameters.

### E. Heat capacity
Integrate \(C_p(T)\) and estimate enthalpy changes.

### Minimum notebook requirements

1. Physical question
2. Mathematical model
3. Dataset
4. Visualization
5. Numerical method
6. Parameter estimation
7. Error/residual analysis
8. Validation
9. Physical interpretation
10. Limitations


# 18. Assessment questions

## Conceptual

1. Difference between variance and standard deviation?
2. Why use \(N-1\) for sample variance?
3. What does covariance measure?
4. Why does correlation not imply causation?
5. Difference between differentiation and integration?
6. Why can very small finite-difference steps be problematic?
7. What condition allows bisection?
8. Difference between interpolation and extrapolation?
9. Why is least squares useful for experimental data?
10. Why avoid explicit matrix inversion in numerical calculations?
11. What is an objective function?
12. How is optimization connected to machine learning?

## Implementation

Write Python functions for:

- sample standard deviation
- central numerical derivative
- trapezoidal integration
- bisection root finding
- linear least squares
- Arrhenius parameter estimation

## Scientific reasoning

For every numerical method, identify:

\[
\boxed{
\text{physical quantity}
\rightarrow
\text{mathematical equation}
\rightarrow
\text{numerical approximation}
\rightarrow
\text{Python implementation}
}
\]


# 19. Take-home assignment

## Numerical Analysis of a Materials Property

You are given a dataset containing temperature and a temperature-dependent materials property.

### Part A — Statistics
Calculate mean, standard deviation, range, percentiles, and uncertainty.

### Part B — Visualization
Produce raw-data and transformed-variable plots.

### Part C — Numerical analysis
Use at least two of:
- differentiation
- integration
- interpolation
- root finding

### Part D — Model fitting
Fit at least two competing mathematical models.

### Part E — Optimization
Define and minimize an error/objective function.

### Part F — Scientific interpretation
Discuss model quality, residuals, uncertainty, limitations, physical plausibility, and whether extrapolation is justified.

### Suggested grading

| Component | Marks |
|---|---:|
| Mathematical formulation | 20 |
| Python implementation | 20 |
| Numerical analysis | 15 |
| Statistics | 10 |
| Visualization | 10 |
| Validation/error analysis | 10 |
| Materials interpretation | 10 |
| Code quality/reproducibility | 5 |
| **Total** | **100** |


# 20. Module 4 — Key takeaways

The mathematical foundations developed here feed directly into the next modules.

\[
\boxed{\text{Statistics}\rightarrow\text{data uncertainty}}
\]

\[
\boxed{\text{Numerical calculus}\rightarrow\text{scientific models}}
\]

\[
\boxed{\text{Least squares}\rightarrow\text{parameter estimation}}
\]

\[
\boxed{\text{Optimization}\rightarrow\text{model calibration}\rightarrow\text{machine learning}}
\]

### Connection to later computational materials work

**Module 4 → Module 5**

- numerical calculus → ODEs
- numerical differentiation → finite differences
- integration → conservation laws
- root finding → nonlinear equations
- optimization → model calibration

**Module 4 → Modules 6–7**

- statistics → preprocessing
- covariance → PCA
- least squares → regression
- optimization → machine learning
- probability → uncertainty and model evaluation

The goal is not to memorize SciPy functions. The goal is to understand **what mathematical problem is being solved, what numerical approximation is being used, how accurate it is, and what the result means physically.**
